# NST Predict C Area + Missing Completion — Full Runner v4 Complete

Bản này sửa lại hoàn chỉnh: train 6v1 + 9v2 có clean progress, checkpoint/resume theo epoch, lưu model đúng đường dẫn Drive, 10v2 load exact path, không còn search model lung tung.


In [ ]:
# =========================
# CELL 1 — CONFIG
# =========================
from pathlib import Path

REPO_URL = "https://github.com/Killua-2002/predict_c_area.git"
PROJECT_DIR = Path("/content/predict_c_area")

DRIVE_ROOT = Path("/content/drive/MyDrive/nst_tach_results")
DRIVE_RESULTS = DRIVE_ROOT / "results_v4_complete"
LOCAL_RESULTS = PROJECT_DIR / "results"

# Chỉ xóa local data cũ trong Colab, KHÔNG xóa Drive checkpoints/models.
CLEAN_LOCAL_DATA = True

# Dataset
NUM_SAMPLES = 5000
TRAIN_POOL_SIZE = 4000
REAL_TEST_SIZE = 1000

# Batch 40 chia hết: train 2800, val 600, test 600, real_test 1000
BATCH_SIZE = 40
VISIBLE_EPOCHS = 70
MISSING_EPOCHS = 90
PATIENCE = 12
VISIBLE_BASE_FILTERS = 32
MISSING_BASE_FILTERS = 32
LR = 1e-4
MAX_KEEP_CHECKPOINTS = 3

print("Repo:", REPO_URL)
print("Project:", PROJECT_DIR)
print("Drive results:", DRIVE_RESULTS)
print("Batch:", BATCH_SIZE, "| visible epochs:", VISIBLE_EPOCHS, "| missing epochs:", MISSING_EPOCHS)


In [ ]:
# =========================
# CELL 2 — ENV CHECK
# =========================
import os, sys, subprocess, shutil, time, json
from pathlib import Path

print("Python:", sys.version)
print("Disk:")
os.system("df -h /content || true")
print("RAM:")
os.system("free -h || true")
print("GPU:")
os.system("nvidia-smi || true")

# Colab thường đã có TensorFlow, chỉ cài thêm thư viện nhẹ nếu thiếu.
os.system("python -m pip install -q --upgrade pillow opencv-python matplotlib pandas scikit-learn tqdm")


In [ ]:
# =========================
# CELL 3 — CLONE GIT
# =========================
import os, shutil
from pathlib import Path

os.chdir("/content")
if PROJECT_DIR.exists():
    print("Removing old project dir:", PROJECT_DIR)
    shutil.rmtree(PROJECT_DIR)

cmd = f"git clone {REPO_URL} {PROJECT_DIR}"
print(cmd)
ret = os.system(cmd)
assert ret == 0, "Git clone failed"

os.chdir(PROJECT_DIR)
print("Current dir:", Path.cwd())
print("Top files:")
os.system("find . -maxdepth 3 -type f | sed 's#^./##' | head -100")


In [ ]:
# =========================
# CELL 4 — MOUNT DRIVE + CLEAN OLD LOCAL OUTPUTS
# =========================
from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount("/content/drive", force_remount=False)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_DIR)

if CLEAN_LOCAL_DATA:
    delete_dirs = ["generated_data", "processed_data_256", "dataset", "results", "result", "checkpoint", "checkpoints"]
    for d in delete_dirs:
        p = PROJECT_DIR / d
        if p.exists():
            print("Deleting local old:", p)
            shutil.rmtree(p)

print("Drive results kept/resume from:", DRIVE_RESULTS)


In [ ]:
%%writefile 2v1_prepare_single_chromosomes.py
"""
2v1_prepare_single_chromosomes.py
Flatten raw single chromosome images into prepared_single_chromosomes/images_rgba.

Input accepted:
    source_data/single_chromosomes/**.png|jpg|jpeg|bmp|tif|tiff

Output:
    prepared_single_chromosomes/images_rgba/*.png

This script is intentionally simple and robust: it converts every readable image to RGBA PNG.
The 3v1 generator will later extract chromosome masks from alpha if present, otherwise from background color.
"""
from __future__ import annotations

from pathlib import Path
from PIL import Image
import shutil

ROOT = Path(__file__).resolve().parent
SOURCE_DIR = ROOT / "source_data" / "single_chromosomes"
OUT_DIR = ROOT / "prepared_single_chromosomes" / "images_rgba"
CLEAR_OLD_OUTPUT = True
VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def main():
    if not SOURCE_DIR.exists():
        raise FileNotFoundError(f"Missing source folder: {SOURCE_DIR}")

    if CLEAR_OLD_OUTPUT and OUT_DIR.exists():
        shutil.rmtree(OUT_DIR)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    paths = [p for p in SOURCE_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VALID_EXTS]
    if not paths:
        raise FileNotFoundError(f"No image files found under: {SOURCE_DIR}")

    ok = 0
    skipped = 0
    for idx, p in enumerate(sorted(paths), start=1):
        try:
            img = Image.open(p).convert("RGBA")
            out_name = f"single_{idx:06d}.png"
            img.save(OUT_DIR / out_name)
            ok += 1
        except Exception as e:
            print(f"[SKIP] {p}: {e}")
            skipped += 1

        if idx % 500 == 0:
            print(f"Processed {idx}/{len(paths)}")

    print("Done preparing single chromosomes.")
    print(f"Input images : {len(paths)}")
    print(f"Saved images : {ok}")
    print(f"Skipped      : {skipped}")
    print(f"Output folder: {OUT_DIR}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile 3v1_generate_synthetic_masks.py
"""
3v1_generate_synthetic_masks.py
Generate 5,000 realistic chromosome overlap samples for missing-part prediction.

Final logic:
- No thick border/contour is drawn into train images or masks.
- The overlap region C is rendered by blending only the top chromosome inside C at 50% opacity
  with soft edge/blur, similar to a Photoshop opacity/filter layer.
- Saves: full masks A/B/C, visible A/B, missing gaps A/B, and top-order labels.
"""
from __future__ import annotations

import argparse
import csv
import random
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageFilter

ROOT = Path(__file__).resolve().parent
SOURCE_DIR = ROOT / "prepared_single_chromosomes" / "images_rgba"
OUT_ROOT = ROOT / "generated_data"
OUT_IMAGE_DIR = OUT_ROOT / "images"
OUT_MASK_A_DIR = OUT_ROOT / "masks_A"
OUT_MASK_B_DIR = OUT_ROOT / "masks_B"
OUT_MASK_C_DIR = OUT_ROOT / "masks_C"
OUT_VISIBLE_A_DIR = OUT_ROOT / "visible_A"
OUT_VISIBLE_B_DIR = OUT_ROOT / "visible_B"
OUT_GAP_A_DIR = OUT_ROOT / "gap_A"
OUT_GAP_B_DIR = OUT_ROOT / "gap_B"
OUT_PREVIEW_DIR = OUT_ROOT / "previews"
OUT_LABEL_CSV = OUT_ROOT / "order_labels.csv"

CANVAS_SIZE = 512
BACKGROUND_DIFF_THRESHOLD = 22
MIN_OBJECT_AREA = 100
MIN_OVERLAP_PIXELS = 120
MAX_OVERLAP_RATIO = 0.55
TARGET_LONG_SIDE_MIN = 250
TARGET_LONG_SIDE_MAX = 380
TOP_OPACITY_IN_C = 0.50
SOFT_EDGE_BLUR_RADIUS = 3.0
NOISE_PROB = 0.30
NOISE_STD = 2.0
VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def reset_output(clear_old: bool) -> None:
    if clear_old and OUT_ROOT.exists():
        shutil.rmtree(OUT_ROOT)
    for folder in [
        OUT_IMAGE_DIR, OUT_MASK_A_DIR, OUT_MASK_B_DIR, OUT_MASK_C_DIR,
        OUT_VISIBLE_A_DIR, OUT_VISIBLE_B_DIR, OUT_GAP_A_DIR, OUT_GAP_B_DIR,
        OUT_PREVIEW_DIR,
    ]:
        folder.mkdir(parents=True, exist_ok=True)


def keep_largest_component(mask: np.ndarray) -> np.ndarray:
    mask_uint8 = mask.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_uint8, connectivity=8)
    if num_labels <= 1:
        return mask
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_label = 1 + int(np.argmax(areas))
    return labels == largest_label


def extract_chromosome_object(image_path: Path):
    img = Image.open(image_path).convert("RGBA")
    arr = np.array(img)
    rgb = arr[:, :, :3]
    alpha = arr[:, :, 3]
    h, w = rgb.shape[:2]

    if np.min(alpha) < 250:
        mask = alpha > 10
    else:
        corner_size = max(5, min(h, w) // 12)
        corners = np.concatenate([
            rgb[:corner_size, :corner_size].reshape(-1, 3),
            rgb[:corner_size, w-corner_size:w].reshape(-1, 3),
            rgb[h-corner_size:h, :corner_size].reshape(-1, 3),
            rgb[h-corner_size:h, w-corner_size:w].reshape(-1, 3),
        ], axis=0)
        bg_color = np.median(corners, axis=0)
        diff = np.linalg.norm(rgb.astype(np.float32) - bg_color.astype(np.float32), axis=2)
        gray = np.mean(rgb, axis=2)
        bg_gray = float(np.mean(bg_color))
        mask = (diff > BACKGROUND_DIFF_THRESHOLD) | (gray < bg_gray - 8)

    mask_uint8 = (mask.astype(np.uint8) * 255)
    kernel = np.ones((3, 3), np.uint8)
    mask_uint8 = cv2.morphologyEx(mask_uint8, cv2.MORPH_OPEN, kernel)
    mask_uint8 = cv2.morphologyEx(mask_uint8, cv2.MORPH_CLOSE, kernel)
    mask = keep_largest_component(mask_uint8 > 0)

    if int(mask.sum()) < MIN_OBJECT_AREA:
        return None

    ys, xs = np.where(mask)
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    # No thick border: only crop exactly around the detected object with tiny safe pad.
    pad = 1
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(w - 1, x2 + pad), min(h - 1, y2 + pad)

    cropped_rgb = rgb[y1:y2+1, x1:x2+1]
    cropped_mask = mask[y1:y2+1, x1:x2+1]

    obj_rgba_arr = np.zeros((cropped_rgb.shape[0], cropped_rgb.shape[1], 4), dtype=np.uint8)
    obj_rgba_arr[:, :, :3] = cropped_rgb
    obj_rgba_arr[:, :, 3] = cropped_mask.astype(np.uint8) * 255

    return (
        image_path,
        Image.fromarray(obj_rgba_arr, "RGBA"),
        Image.fromarray((cropped_mask.astype(np.uint8) * 255), "L"),
    )


def load_source_objects(progress_every: int = 500):
    paths = sorted([p for p in SOURCE_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VALID_EXTS])
    if len(paths) < 2:
        raise ValueError(f"Need at least 2 chromosome images in {SOURCE_DIR}")
    print(f"[3v1] Found {len(paths)} prepared single chromosome images.")
    print("[3v1] Caching extracted chromosome objects into RAM for faster generation...")
    objects = []
    skipped = 0
    t0 = time.time()
    for i, p in enumerate(paths, 1):
        try:
            item = extract_chromosome_object(p)
            if item is None:
                skipped += 1
            else:
                objects.append(item)
        except Exception as e:
            skipped += 1
            if skipped <= 10:
                print(f"[SKIP] {p.name}: {e}")
        if i % progress_every == 0 or i == len(paths):
            print(f"[3v1][cache] {i}/{len(paths)} | ok={len(objects)} | skipped={skipped} | elapsed={time.time()-t0:.1f}s", flush=True)
    if len(objects) < 2:
        raise RuntimeError("Not enough valid chromosome objects after extraction.")
    return objects


def resize_keep_ratio(img: Image.Image, mask: Image.Image, target_long_side: int):
    w, h = img.size
    scale = target_long_side / max(w, h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return img.resize((new_w, new_h), Image.BILINEAR), mask.resize((new_w, new_h), Image.NEAREST)


def rotate_pair(img: Image.Image, mask: Image.Image, angle: float):
    img_rot = img.rotate(angle, expand=True, resample=Image.BILINEAR, fillcolor=(255, 255, 255, 0))
    mask_rot = mask.rotate(angle, expand=True, resample=Image.NEAREST, fillcolor=0)
    return img_rot, mask_rot


def paste_to_rgba_canvas(obj_rgba: Image.Image, obj_mask: Image.Image, center_x: int, center_y: int):
    layer = Image.new("RGBA", (CANVAS_SIZE, CANVAS_SIZE), (255, 255, 255, 0))
    mask_canvas = Image.new("L", (CANVAS_SIZE, CANVAS_SIZE), 0)
    w, h = obj_rgba.size
    x = int(center_x - w / 2)
    y = int(center_y - h / 2)
    layer.alpha_composite(obj_rgba.convert("RGBA"), dest=(x, y))

    mask_arr = np.array(mask_canvas)
    obj_mask_arr = np.array(obj_mask.convert("L"))
    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(CANVAS_SIZE, x + w), min(CANVAS_SIZE, y + h)
    ox1, oy1 = x1 - x, y1 - y
    ox2, oy2 = ox1 + (x2 - x1), oy1 + (y2 - y1)
    if x1 < x2 and y1 < y2:
        region = mask_arr[y1:y2, x1:x2]
        obj_region = obj_mask_arr[oy1:oy2, ox1:ox2]
        region[obj_region > 0] = 255
        mask_arr[y1:y2, x1:x2] = region
    return layer, Image.fromarray(mask_arr, "L")


def alpha_over(base_rgb: np.ndarray, top_rgba: np.ndarray, alpha_override: np.ndarray | None = None) -> np.ndarray:
    base = base_rgb.astype(np.float32)
    top_rgb = top_rgba[:, :, :3].astype(np.float32)
    alpha = top_rgba[:, :, 3].astype(np.float32) / 255.0
    if alpha_override is not None:
        alpha = alpha * np.clip(alpha_override.astype(np.float32), 0.0, 1.0)
    alpha = alpha[:, :, None]
    return np.clip(top_rgb * alpha + base * (1.0 - alpha), 0, 255).astype(np.uint8)


def composite_realistic(layer_A: Image.Image, layer_B: Image.Image, mask_A: Image.Image, mask_B: Image.Image, top_label: str):
    A = np.array(mask_A) > 0
    B = np.array(mask_B) > 0
    C = A & B
    base = np.ones((CANVAS_SIZE, CANVAS_SIZE, 3), dtype=np.uint8) * 255
    arr_A = np.array(layer_A.convert("RGBA"))
    arr_B = np.array(layer_B.convert("RGBA"))
    lower_rgba, top_rgba = (arr_B, arr_A) if top_label == "A_ON_TOP" else (arr_A, arr_B)

    # 1) lower chromosome fully visible
    out = alpha_over(base, lower_rgba)

    # 2) only inside C, blend the top chromosome at 50% opacity with soft boundary
    c_soft = np.array(
        Image.fromarray((C.astype(np.uint8) * 255), "L").filter(ImageFilter.GaussianBlur(radius=SOFT_EDGE_BLUR_RADIUS))
    ).astype(np.float32) / 255.0
    alpha_factor = 1.0 - c_soft * (1.0 - TOP_OPACITY_IN_C)

    # slight texture blur inside C to reduce fake hard crossing edge
    top_blur = np.array(Image.fromarray(top_rgba[:, :, :3], "RGB").filter(ImageFilter.GaussianBlur(radius=1.15))).astype(np.uint8)
    c3 = c_soft[:, :, None]
    top_rgba_soft = top_rgba.copy()
    top_rgba_soft[:, :, :3] = np.clip(top_rgba[:, :, :3] * (1 - c3) + top_blur * c3, 0, 255).astype(np.uint8)
    out = alpha_over(out, top_rgba_soft, alpha_override=alpha_factor)

    if random.random() < NOISE_PROB:
        out = np.clip(out.astype(np.float32) + np.random.normal(0, NOISE_STD, out.shape), 0, 255).astype(np.uint8)
    return Image.fromarray(out, "RGB"), C


def make_preview(image: Image.Image, mask_A: Image.Image, mask_B: Image.Image, mask_C: Image.Image):
    base = np.array(image.convert("RGB")).astype(np.float32)
    A = np.array(mask_A) > 0
    B = np.array(mask_B) > 0
    C = np.array(mask_C) > 0
    overlay = base.copy()
    overlay[A] = overlay[A] * 0.70 + np.array([255, 0, 0]) * 0.30
    overlay[B] = overlay[B] * 0.70 + np.array([0, 255, 0]) * 0.30
    overlay[C] = overlay[C] * 0.50 + np.array([255, 255, 0]) * 0.50
    return Image.fromarray(np.clip(overlay, 0, 255).astype(np.uint8), "RGB")


def make_sample(item_A, item_B):
    path_A, obj_A, mask_A = item_A
    path_B, obj_B, mask_B = item_B
    target_A = random.randint(TARGET_LONG_SIDE_MIN, TARGET_LONG_SIDE_MAX)
    target_B = random.randint(TARGET_LONG_SIDE_MIN, TARGET_LONG_SIDE_MAX)
    obj_A, mask_A = resize_keep_ratio(obj_A, mask_A, target_A)
    obj_B, mask_B = resize_keep_ratio(obj_B, mask_B, target_B)
    obj_A, mask_A = rotate_pair(obj_A, mask_A, 90 + random.uniform(-10, 10))
    obj_B, mask_B = rotate_pair(obj_B, mask_B, random.uniform(-10, 10))

    center_x = CANVAS_SIZE // 2 + random.randint(-16, 16)
    center_y = CANVAS_SIZE // 2 + random.randint(-16, 16)
    A_cx, A_cy = center_x + random.randint(-20, 20), center_y + random.randint(-12, 12)
    B_cx, B_cy = center_x + random.randint(-12, 12), center_y + random.randint(-20, 20)
    layer_A, mask_A_canvas = paste_to_rgba_canvas(obj_A, mask_A, A_cx, A_cy)
    layer_B, mask_B_canvas = paste_to_rgba_canvas(obj_B, mask_B, B_cx, B_cy)

    A_arr = np.array(mask_A_canvas) > 0
    B_arr = np.array(mask_B_canvas) > 0
    C_arr = A_arr & B_arr
    overlap_pixels = int(C_arr.sum())
    area_A, area_B = max(1, int(A_arr.sum())), max(1, int(B_arr.sum()))
    overlap_ratio = overlap_pixels / min(area_A, area_B)
    if overlap_pixels < MIN_OVERLAP_PIXELS or overlap_ratio > MAX_OVERLAP_RATIO:
        return None

    top_label = "A_ON_TOP" if random.random() < 0.5 else "B_ON_TOP"
    final_image, C_arr = composite_realistic(layer_A, layer_B, mask_A_canvas, mask_B_canvas, top_label)
    if top_label == "A_ON_TOP":
        visible_A = A_arr
        visible_B = B_arr & (~C_arr)
        gap_A = np.zeros_like(C_arr)
        gap_B = C_arr
    else:
        visible_A = A_arr & (~C_arr)
        visible_B = B_arr
        gap_A = C_arr
        gap_B = np.zeros_like(C_arr)

    return {
        "image": final_image,
        "mask_A": mask_A_canvas,
        "mask_B": mask_B_canvas,
        "mask_C": Image.fromarray((C_arr.astype(np.uint8) * 255), "L"),
        "visible_A": Image.fromarray((visible_A.astype(np.uint8) * 255), "L"),
        "visible_B": Image.fromarray((visible_B.astype(np.uint8) * 255), "L"),
        "gap_A": Image.fromarray((gap_A.astype(np.uint8) * 255), "L"),
        "gap_B": Image.fromarray((gap_B.astype(np.uint8) * 255), "L"),
        "source_A": path_A.name,
        "source_B": path_B.name,
        "top_label": top_label,
        "top_class": 0 if top_label == "A_ON_TOP" else 1,
        "overlap_pixels": overlap_pixels,
        "overlap_ratio": overlap_ratio,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--num-samples", type=int, default=5000)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--clear-old", action="store_true", default=True)
    ap.add_argument("--no-clear-old", dest="clear_old", action="store_false")
    ap.add_argument("--progress-every", type=int, default=100)
    ap.add_argument("--max-attempts-factor", type=int, default=80)
    args = ap.parse_args()

    random.seed(args.seed)
    np.random.seed(args.seed)
    reset_output(args.clear_old)

    print("=" * 80)
    print("3v1 GENERATE REALISTIC SYNTHETIC OVERLAP DATASET")
    print("=" * 80)
    print(f"Samples target : {args.num_samples}")
    print(f"Output folder  : {OUT_ROOT}")
    print("Logic          : no contour; C region uses 50% opacity top-layer blend")

    objects = load_source_objects(progress_every=max(100, args.progress_every))
    rows = []
    created = 0
    attempts = 0
    max_attempts = args.num_samples * args.max_attempts_factor
    t0 = time.time()

    while created < args.num_samples and attempts < max_attempts:
        attempts += 1
        item_A, item_B = random.sample(objects, 2)
        sample = make_sample(item_A, item_B)
        if sample is None:
            continue
        created += 1
        name = f"img_{created:06d}.png"
        sample["image"].save(OUT_IMAGE_DIR / name)
        sample["mask_A"].save(OUT_MASK_A_DIR / name)
        sample["mask_B"].save(OUT_MASK_B_DIR / name)
        sample["mask_C"].save(OUT_MASK_C_DIR / name)
        sample["visible_A"].save(OUT_VISIBLE_A_DIR / name)
        sample["visible_B"].save(OUT_VISIBLE_B_DIR / name)
        sample["gap_A"].save(OUT_GAP_A_DIR / name)
        sample["gap_B"].save(OUT_GAP_B_DIR / name)
        make_preview(sample["image"], sample["mask_A"], sample["mask_B"], sample["mask_C"]).save(OUT_PREVIEW_DIR / name)
        rows.append({
            "filename": name,
            "source_A": sample["source_A"],
            "source_B": sample["source_B"],
            "top_label": sample["top_label"],
            "top_class": sample["top_class"],
            "overlap_pixels": sample["overlap_pixels"],
            "overlap_ratio": round(float(sample["overlap_ratio"]), 6),
        })
        if created % args.progress_every == 0 or created == args.num_samples:
            rate = created / max(time.time() - t0, 1e-6)
            print(f"[3v1] created={created}/{args.num_samples} attempts={attempts} rate={rate:.2f} img/s elapsed={time.time()-t0:.1f}s", flush=True)

    with open(OUT_LABEL_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["filename", "source_A", "source_B", "top_label", "top_class", "overlap_pixels", "overlap_ratio"])
        writer.writeheader()
        writer.writerows(rows)

    print("=" * 80)
    print("3v1 DONE")
    print("=" * 80)
    print(f"Created : {created}")
    print(f"Attempts: {attempts}")
    print(f"Labels  : {OUT_LABEL_CSV}")
    if created < args.num_samples:
        raise RuntimeError(f"Only created {created}/{args.num_samples}. Try lowering MIN_OVERLAP_PIXELS or increasing --max-attempts-factor.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile 4v1_preprocess_to_256.py
"""
4v1_preprocess_to_256.py
Resize generated realistic overlap dataset to 256x256.

Supports extended masks:
- masks_A, masks_B, masks_C
- visible_A, visible_B
- gap_A, gap_B
Copies order_labels.csv to processed_data_256.
"""
from pathlib import Path
from PIL import Image
import numpy as np
import shutil

ROOT = Path(__file__).resolve().parent
INPUT_ROOT = ROOT / "generated_data"
OUTPUT_ROOT = ROOT / "processed_data_256"
TARGET_SIZE = 256
CLEAR_OLD_OUTPUT = True

IMAGE_SUBDIR = "images"
MASK_SUBDIRS = ["masks_A", "masks_B", "masks_C", "visible_A", "visible_B", "gap_A", "gap_B"]

if CLEAR_OLD_OUTPUT and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

(OUTPUT_ROOT / IMAGE_SUBDIR).mkdir(parents=True, exist_ok=True)
for sub in MASK_SUBDIRS:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)


def resize_with_padding(img, target_size=256, is_mask=False):
    w, h = img.size
    scale = min(target_size / w, target_size / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    if is_mask:
        img = img.resize((new_w, new_h), Image.NEAREST)
        canvas = Image.new("L", (target_size, target_size), 0)
    else:
        img = img.resize((new_w, new_h), Image.BILINEAR)
        canvas = Image.new("L", (target_size, target_size), 255)
    paste_x = (target_size - new_w) // 2
    paste_y = (target_size - new_h) // 2
    canvas.paste(img, (paste_x, paste_y))
    return canvas


def binarize_mask(mask_img):
    arr = np.array(mask_img)
    arr = (arr > 127).astype(np.uint8) * 255
    return Image.fromarray(arr, mode="L")

image_paths = sorted((INPUT_ROOT / IMAGE_SUBDIR).glob("*.png"))
print(f"Found {len(image_paths)} generated images.")
if not image_paths:
    raise FileNotFoundError("No generated images found. Run 3v1_generate_synthetic_masks.py first.")

for idx, img_path in enumerate(image_paths, start=1):
    name = img_path.name

    missing = [sub for sub in MASK_SUBDIRS if not (INPUT_ROOT / sub / name).exists()]
    if missing:
        print(f"[SKIP] Missing {missing} for {name}")
        continue

    img = Image.open(img_path).convert("L")
    img = resize_with_padding(img, target_size=TARGET_SIZE, is_mask=False)
    img.save(OUTPUT_ROOT / IMAGE_SUBDIR / name)

    for sub in MASK_SUBDIRS:
        mask = Image.open(INPUT_ROOT / sub / name).convert("L")
        mask = binarize_mask(mask)
        mask = resize_with_padding(mask, target_size=TARGET_SIZE, is_mask=True)
        mask = binarize_mask(mask)
        mask.save(OUTPUT_ROOT / sub / name)

    if idx % 100 == 0:
        print(f"Processed {idx}/{len(image_paths)}")

label_csv = INPUT_ROOT / "order_labels.csv"
if label_csv.exists():
    shutil.copy2(label_csv, OUTPUT_ROOT / "order_labels.csv")
    print(f"Copied labels: {OUTPUT_ROOT / 'order_labels.csv'}")

print("Done preprocessing to 256x256.")


In [ ]:
%%writefile 5v1_split_data.py
"""
5v1_split_data.py
Split 5,000 processed samples into:
- 4,000 model-development samples -> train/val/test
- 1,000 real_test samples kept untouched for final evaluation

Default 4,000 split:
- train = 2,800
- val   = 600
- test  = 600
- real_test = 1,000
"""
from pathlib import Path
import shutil
import random
import csv

CLEAR_OLD_DATASET = True
ROOT = Path(__file__).resolve().parent
PROCESSED = ROOT / "processed_data_256"
IMAGE_DIR = PROCESSED / "images"
DATASET_DIR = ROOT / "dataset"

MASK_SUBDIRS = ["masks_A", "masks_B", "masks_C", "visible_A", "visible_B", "gap_A", "gap_B"]

TRAIN_POOL_SIZE = 4000
REAL_TEST_SIZE = 1000
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42
random.seed(SEED)

if CLEAR_OLD_DATASET and DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

splits = ["train", "val", "test", "real_test"]
for split in splits:
    (DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    for sub in MASK_SUBDIRS:
        (DATASET_DIR / split / sub).mkdir(parents=True, exist_ok=True)

image_paths = sorted(IMAGE_DIR.glob("*.png"))
names = [p.name for p in image_paths]
if not names:
    raise FileNotFoundError("No processed images found. Run 4v1_preprocess_to_256.py first.")

random.shuffle(names)

if len(names) < TRAIN_POOL_SIZE + REAL_TEST_SIZE:
    raise ValueError(f"Need at least {TRAIN_POOL_SIZE + REAL_TEST_SIZE} images, found {len(names)}. Run 3v1 with NUM_SAMPLES=5000.")

train_pool = names[:TRAIN_POOL_SIZE]
real_test_names = names[TRAIN_POOL_SIZE:TRAIN_POOL_SIZE + REAL_TEST_SIZE]

n_train = int(TRAIN_POOL_SIZE * TRAIN_RATIO)
n_val = int(TRAIN_POOL_SIZE * VAL_RATIO)
train_names = train_pool[:n_train]
val_names = train_pool[n_train:n_train + n_val]
test_names = train_pool[n_train + n_val:]

print(f"Total processed: {len(names)}")
print(f"Train pool     : {len(train_pool)}")
print(f"  train        : {len(train_names)}")
print(f"  val          : {len(val_names)}")
print(f"  test         : {len(test_names)}")
print(f"Real test hold : {len(real_test_names)}")

# Read labels metadata if exists
label_map = {}
label_csv = PROCESSED / "order_labels.csv"
if label_csv.exists():
    with open(label_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            label_map[row["filename"]] = row


def copy_set(file_names, split_name):
    split_rows = []
    for name in file_names:
        shutil.copy2(PROCESSED / "images" / name, DATASET_DIR / split_name / "images" / name)
        for sub in MASK_SUBDIRS:
            src = PROCESSED / sub / name
            if not src.exists():
                raise FileNotFoundError(f"Missing {src}")
            shutil.copy2(src, DATASET_DIR / split_name / sub / name)
        if name in label_map:
            split_rows.append(label_map[name])

    if split_rows:
        out_csv = DATASET_DIR / split_name / "order_labels.csv"
        with open(out_csv, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(split_rows[0].keys()))
            writer.writeheader()
            writer.writerows(split_rows)
        print(f"Saved labels: {out_csv}")

copy_set(train_names, "train")
copy_set(val_names, "val")
copy_set(test_names, "test")
copy_set(real_test_names, "real_test")

# master split manifest
with open(DATASET_DIR / "split_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["split", "count"])
    writer.writerow(["train", len(train_names)])
    writer.writerow(["val", len(val_names)])
    writer.writerow(["test", len(test_names)])
    writer.writerow(["real_test", len(real_test_names)])

print("Done splitting dataset.")


In [ ]:
%%writefile 6v1_train_visible_order.py
"""
6v1_train_visible_order.py
Model 1: predict visible A/B/C masks and classify which chromosome is on top.

Dataset expected:
    dataset/train|val|test|real_test/
        images/
        visible_A/
        visible_B/
        masks_C/
        order_labels.csv   with top_class: 0=A_ON_TOP, 1=B_ON_TOP

Output:
    results/visible_order/
        best_visible_order_teacher.keras
        history_visible_order.csv/json
        metrics_visible_order_real_test.json
        confusion_order_heatmap.png
        visible_order_showcase.png
        checkpoints/epoch_XXX.weights.h5 for resume
"""
from __future__ import annotations

import argparse, csv, json, os, random, shutil, math, time, re
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
import tensorflow as tf
from tensorflow import keras
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

tf.get_logger().setLevel("ERROR")
try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
except Exception:
    pass
from tensorflow.keras import layers

IMG_SIZE = 256
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def read_labels(split_dir: Path) -> Dict[str, int]:
    csv_path = split_dir / "order_labels.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing {csv_path}")
    labels = {}
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            labels[row["filename"]] = int(row["top_class"])
    return labels


def get_split_lists(dataset_dir: Path, split: str):
    split_dir = dataset_dir / split
    labels = read_labels(split_dir)
    image_paths, va_paths, vb_paths, c_paths, y_order = [], [], [], [], []
    for p in sorted((split_dir / "images").glob("*.png")):
        name = p.name
        va = split_dir / "visible_A" / name
        vb = split_dir / "visible_B" / name
        mc = split_dir / "masks_C" / name
        if name in labels and va.exists() and vb.exists() and mc.exists():
            image_paths.append(str(p)); va_paths.append(str(va)); vb_paths.append(str(vb)); c_paths.append(str(mc)); y_order.append(labels[name])
    if not image_paths:
        raise FileNotFoundError(f"No valid samples in {split_dir}")
    return image_paths, va_paths, vb_paths, c_paths, np.array(y_order, dtype=np.int32)


def read_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear")
    img.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return img


def read_mask(path):
    m = tf.io.read_file(path)
    m = tf.image.decode_png(m, channels=1)
    m = tf.image.resize(m, (IMG_SIZE, IMG_SIZE), method="nearest")
    m = tf.cast(m > 127, tf.float32)
    m.set_shape([IMG_SIZE, IMG_SIZE, 1])
    return m


def load_sample(img_p, va_p, vb_p, c_p, top_class):
    x = read_image(img_p)
    va, vb, c = read_mask(va_p), read_mask(vb_p), read_mask(c_p)
    y_seg = tf.concat([va, vb, c], axis=-1)
    y_seg.set_shape([IMG_SIZE, IMG_SIZE, 3])
    y_order = tf.one_hot(tf.cast(top_class, tf.int32), 2)
    return x, {"seg": y_seg, "order": y_order}


def augment(x, y):
    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_left_right(x)
        y["seg"] = tf.image.flip_left_right(y["seg"])
    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_up_down(x)
        y["seg"] = tf.image.flip_up_down(y["seg"])
    if tf.random.uniform(()) > 0.7:
        x = tf.clip_by_value(x + tf.random.normal(tf.shape(x), 0, 0.015), 0, 1)
    return x, y


def make_ds(dataset_dir: Path, split: str, batch: int, shuffle=False, do_aug=False):
    img, va, vb, c, order = get_split_lists(dataset_dir, split)
    ds = tf.data.Dataset.from_tensor_slices((img, va, vb, c, order))
    if shuffle:
        ds = ds.shuffle(min(len(img), 4096), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)
    if do_aug:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(1), len(img)


def conv_block(x, f, drop=0.0):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(f, 3, padding="same", use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    if drop: x = layers.SpatialDropout2D(drop)(x)
    return x


def build_model(base=32):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 1), name="image")
    s1 = conv_block(inp, base); p1 = layers.MaxPooling2D()(s1)
    s2 = conv_block(p1, base*2); p2 = layers.MaxPooling2D()(s2)
    s3 = conv_block(p2, base*4, 0.05); p3 = layers.MaxPooling2D()(s3)
    s4 = conv_block(p3, base*8, 0.10); p4 = layers.MaxPooling2D()(s4)
    b = conv_block(p4, base*16, 0.15)

    # order classifier from bottleneck
    o = layers.GlobalAveragePooling2D()(b)
    o = layers.Dense(128, activation="relu")(o)
    o = layers.Dropout(0.25)(o)
    order = layers.Dense(2, activation="softmax", name="order")(o)

    x = layers.Conv2DTranspose(base*8, 2, strides=2, padding="same")(b); x = layers.Concatenate()([x, s4]); x = conv_block(x, base*8, 0.10)
    x = layers.Conv2DTranspose(base*4, 2, strides=2, padding="same")(x); x = layers.Concatenate()([x, s3]); x = conv_block(x, base*4, 0.05)
    x = layers.Conv2DTranspose(base*2, 2, strides=2, padding="same")(x); x = layers.Concatenate()([x, s2]); x = conv_block(x, base*2)
    x = layers.Conv2DTranspose(base, 2, strides=2, padding="same")(x); x = layers.Concatenate()([x, s1]); x = conv_block(x, base)
    seg = layers.Conv2D(3, 1, activation="sigmoid", dtype="float32", name="seg")(x)
    return keras.Model(inp, {"seg": seg, "order": order}, name="Visible_AB_C_Order_Teacher")


def dice_metric(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * y_pred, axis=[1, 2])
    den = tf.reduce_sum(y_true + y_pred, axis=[1, 2])
    return tf.reduce_mean((2*inter + 1e-6) / (den + 1e-6))


def dice_loss(y_true, y_pred):
    inter = tf.reduce_sum(y_true * y_pred, axis=[1,2])
    den = tf.reduce_sum(y_true + y_pred, axis=[1,2])
    dice = (2*inter + 1e-6) / (den + 1e-6)
    return 1.0 - tf.reduce_mean(dice)


def seg_loss(y_true, y_pred):
    bce = keras.backend.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)


def save_history(history, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    hist = {k: [float(x) for x in v] for k, v in history.history.items()}
    with open(out_dir / "history_visible_order.json", "w", encoding="utf-8") as f:
        json.dump(hist, f, indent=2)
    keys = list(hist.keys())
    with open(out_dir / "history_visible_order.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["epoch"] + keys)
        for i in range(len(next(iter(hist.values())))):
            w.writerow([i+1] + [hist[k][i] for k in keys])

    for metric_name, title in [("loss", "Total loss"), ("seg_dice_metric", "Segmentation Dice"), ("order_accuracy", "Order accuracy")]:
        plt.figure(figsize=(7,4))
        if metric_name in hist: plt.plot(hist[metric_name], label="train")
        if "val_" + metric_name in hist: plt.plot(hist["val_" + metric_name], label="val")
        plt.title(title); plt.xlabel("Epoch"); plt.ylabel(title); plt.legend(); plt.tight_layout()
        plt.savefig(out_dir / f"curve_{metric_name}.png", dpi=160); plt.close()


def np_load_gray(path):
    arr = np.array(Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
    return arr[..., None]

def np_load_mask(path):
    arr = np.array(Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE))).astype(np.float32)
    return (arr > 127).astype(np.float32)


def evaluate_real_test(model, dataset_dir: Path, out_dir: Path, batch_size: int = 16, max_showcase: int = 60):
    out_dir.mkdir(parents=True, exist_ok=True)
    split_dir = dataset_dir / "real_test"
    labels = read_labels(split_dir)
    names = [p.name for p in sorted((split_dir / "images").glob("*.png")) if p.name in labels]
    X, Y_seg, Y_order = [], [], []
    for n in names:
        X.append(np_load_gray(split_dir / "images" / n))
        va = np_load_mask(split_dir / "visible_A" / n)
        vb = np_load_mask(split_dir / "visible_B" / n)
        c = np_load_mask(split_dir / "masks_C" / n)
        Y_seg.append(np.stack([va, vb, c], axis=-1))
        Y_order.append(labels[n])
    X = np.stack(X).astype(np.float32)
    Y_seg = np.stack(Y_seg).astype(np.float32)
    Y_order = np.array(Y_order, dtype=np.int32)

    print(f"[6v1][eval] Predicting real_test: {len(names)} images ...")
    pred = model.predict(X, batch_size=batch_size, verbose=0)
    pred_seg_prob, pred_order_prob = unpack_visible_order_prediction(pred)
    pred_seg = (pred_seg_prob >= 0.5).astype(np.float32)
    pred_order = np.argmax(pred_order_prob, axis=1)

    dice_per_ch = []
    for c in range(3):
        inter = np.sum(Y_seg[...,c] * pred_seg[...,c])
        den = np.sum(Y_seg[...,c] + pred_seg[...,c])
        dice_per_ch.append(float((2*inter + 1e-7) / (den + 1e-7)))
    order_acc = float(np.mean(pred_order == Y_order))
    mean_dice = float(np.mean(dice_per_ch))

    conf = np.zeros((2,2), dtype=int)
    for t, p in zip(Y_order, pred_order): conf[t,p] += 1
    conf_norm = conf / np.maximum(conf.sum(axis=1, keepdims=True), 1)

    metrics = {
        "real_test_count": int(len(names)),
        "mean_visible_seg_dice_percent": mean_dice*100,
        "dice_visible_A_percent": dice_per_ch[0]*100,
        "dice_visible_B_percent": dice_per_ch[1]*100,
        "dice_C_percent": dice_per_ch[2]*100,
        "order_accuracy_percent": order_acc*100,
        "confusion_matrix": conf.tolist(),
    }
    with open(out_dir / "metrics_visible_order_real_test.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    plt.figure(figsize=(5,4))
    plt.imshow(conf_norm*100)
    plt.xticks([0,1], ["A_TOP", "B_TOP"]); plt.yticks([0,1], ["A_TOP", "B_TOP"])
    plt.xlabel("Predicted"); plt.ylabel("Ground truth"); plt.title("Order classifier confusion (%)")
    for i in range(2):
        for j in range(2): plt.text(j, i, f"{conf_norm[i,j]*100:.1f}%", ha="center", va="center")
    plt.colorbar(label="Percent"); plt.tight_layout(); plt.savefig(out_dir / "confusion_order_heatmap.png", dpi=180); plt.close()

    # showcase: 60 samples, each row: overlap | visible A pred | visible B pred | C pred | order
    show_dir = out_dir / "showcase"; show_dir.mkdir(exist_ok=True)
    n_show = min(max_showcase, len(names))
    cols = 5; rows = n_show
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2.2, max(8, rows*1.35)))
    if rows == 1: axes = np.expand_dims(axes, 0)
    for i in range(n_show):
        gray = X[i,...,0]
        imgs = [gray, pred_seg[i,...,0], pred_seg[i,...,1], pred_seg[i,...,2]]
        titles = ["Overlap", "Pred A_visible", "Pred B_visible", "Pred C"]
        for j in range(4):
            axes[i,j].imshow(imgs[j], cmap="gray"); axes[i,j].axis("off")
            if i == 0: axes[i,j].set_title(titles[j], fontsize=8)
        txt = "A_ON_TOP" if pred_order[i] == 0 else "B_ON_TOP"
        gt = "A_ON_TOP" if Y_order[i] == 0 else "B_ON_TOP"
        axes[i,4].axis("off"); axes[i,4].text(0.0, 0.5, f"GT: {gt}\nPred: {txt}", fontsize=7)
    plt.tight_layout(); plt.savefig(show_dir / "visible_order_showcase_60.png", dpi=180); plt.close()

    return metrics



def unpack_visible_order_prediction(pred):
    """Return (seg_prob, order_prob) for Keras dict/list outputs."""
    if isinstance(pred, dict):
        return pred["seg"], pred["order"]
    if isinstance(pred, (list, tuple)):
        seg = None
        order = None
        for item in pred:
            arr = np.asarray(item)
            if arr.ndim == 4 and arr.shape[-1] == 3:
                seg = item
            elif arr.ndim == 2 and arr.shape[-1] == 2:
                order = item
        if seg is not None and order is not None:
            return seg, order
    raise ValueError(f"Cannot unpack visible/order prediction outputs: {type(pred)}")


def checkpoint_epoch(path: Path) -> int:
    m = re.search(r"epoch_(\d+)\.weights\.h5$", path.name)
    return int(m.group(1)) if m else -1


def latest_weight_checkpoint(ckpt_dir: Path):
    files = sorted(ckpt_dir.glob("epoch_*.weights.h5"), key=checkpoint_epoch)
    files = [p for p in files if checkpoint_epoch(p) > 0]
    if not files:
        return 0, None
    p = files[-1]
    return checkpoint_epoch(p), p


class KeepLastCheckpoints(keras.callbacks.Callback):
    def __init__(self, ckpt_dir: Path, keep: int = 3):
        super().__init__()
        self.ckpt_dir = Path(ckpt_dir)
        self.keep = int(max(1, keep))

    def on_epoch_end(self, epoch, logs=None):
        files = sorted(self.ckpt_dir.glob("epoch_*.weights.h5"), key=checkpoint_epoch)
        extra = files[:-self.keep]
        for p in extra:
            try:
                p.unlink()
            except Exception:
                pass


class CleanProgressCallback(keras.callbacks.Callback):
    """Clean notebook-friendly progress for 6v1.

    - Hides Keras' noisy default progress output.
    - Shows one tqdm bar per epoch in Colab/Jupyter.
    - Prints one compact metric line after each epoch.
    """
    def __init__(self, total_epochs: int, steps_per_epoch: int, mode: str = "tqdm"):
        super().__init__()
        self.total_epochs = int(total_epochs)
        self.steps_per_epoch = int(max(1, steps_per_epoch))
        self.mode = mode
        self.pbar = None
        self.t0 = None

    @staticmethod
    def _get(logs, key, default=None):
        if not logs:
            return default
        v = logs.get(key, default)
        try:
            return float(v)
        except Exception:
            return default

    @staticmethod
    def _fmt(v, percent=False):
        if v is None:
            return "-"
        return f"{v*100:.2f}%" if percent else f"{v:.4f}"

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()
        desc = f"[6v1] Epoch {epoch+1:03d}/{self.total_epochs:03d}"
        if self.mode == "tqdm" and tqdm is not None:
            self.pbar = tqdm(
                total=self.steps_per_epoch,
                desc=desc,
                unit="batch",
                leave=False,
                dynamic_ncols=True,
                mininterval=0.5,
            )
        else:
            print(desc)

    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}
        if self.pbar is not None:
            self.pbar.update(1)
            self.pbar.set_postfix({
                "loss": self._fmt(self._get(logs, "loss")),
                "dice": self._fmt(self._get(logs, "seg_dice_metric"), percent=True),
                "ord": self._fmt(self._get(logs, "order_accuracy"), percent=True),
            })
        elif self.mode == "line":
            step = batch + 1
            if step == 1 or step == self.steps_per_epoch or step % max(1, self.steps_per_epoch // 5) == 0:
                pct = step / self.steps_per_epoch * 100
                print(
                    f"  step {step:03d}/{self.steps_per_epoch:03d} ({pct:5.1f}%) | "
                    f"loss={self._fmt(self._get(logs, 'loss'))} | "
                    f"dice={self._fmt(self._get(logs, 'seg_dice_metric'), percent=True)} | "
                    f"order={self._fmt(self._get(logs, 'order_accuracy'), percent=True)}"
                )

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if self.pbar is not None:
            self.pbar.close()
            self.pbar = None
        sec = time.time() - self.t0 if self.t0 else 0.0
        msg = (
            f"[6v1][{epoch+1:03d}/{self.total_epochs:03d}] "
            f"{sec:6.1f}s | "
            f"loss={self._fmt(self._get(logs, 'loss'))} | "
            f"val_loss={self._fmt(self._get(logs, 'val_loss'))} | "
            f"dice={self._fmt(self._get(logs, 'seg_dice_metric'), percent=True)} | "
            f"val_dice={self._fmt(self._get(logs, 'val_seg_dice_metric'), percent=True)} | "
            f"order={self._fmt(self._get(logs, 'order_accuracy'), percent=True)} | "
            f"val_order={self._fmt(self._get(logs, 'val_order_accuracy'), percent=True)}"
        )
        print(msg, flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset-dir", default="dataset")
    ap.add_argument("--results-dir", default="results")
    ap.add_argument("--epochs", type=int, default=80)
    ap.add_argument("--batch-size", type=int, default=24)
    ap.add_argument("--base-filters", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--patience", type=int, default=10)
    ap.add_argument("--progress-mode", choices=["tqdm", "line", "keras"], default="tqdm")
    ap.add_argument("--show-summary", action="store_true")
    ap.add_argument("--resume", action="store_true", help="Resume from exact epoch_XXX.weights.h5 checkpoint if available")
    ap.add_argument("--force-restart", action="store_true", help="Ignore checkpoints and train from scratch")
    ap.add_argument("--max-keep-checkpoints", type=int, default=3)
    args = ap.parse_args()

    dataset_dir = Path(args.dataset_dir)
    out_dir = Path(args.results_dir) / "visible_order"
    out_dir.mkdir(parents=True, exist_ok=True)

    train_ds, n_train = make_ds(dataset_dir, "train", args.batch_size, shuffle=True, do_aug=True)
    val_ds, n_val = make_ds(dataset_dir, "val", args.batch_size)
    steps_per_epoch = math.ceil(n_train / args.batch_size)
    val_steps = math.ceil(n_val / args.batch_size)
    print("="*80)
    print("6v1 VISIBLE A/B/C + ORDER TRAINING")
    print("="*80)
    print(f"Dataset: train={n_train}, val={n_val}")
    print(f"Batch={args.batch_size} | steps/epoch={steps_per_epoch} | val_steps={val_steps}")
    print(f"Epochs={args.epochs} | patience={args.patience} | base_filters={args.base_filters} | lr={args.lr}")
    print(f"Progress mode: {args.progress_mode}")

    model = build_model(base=args.base_filters)
    model.compile(
        optimizer=keras.optimizers.Adam(args.lr),
        loss={"seg": seg_loss, "order": "categorical_crossentropy"},
        loss_weights={"seg": 1.0, "order": 0.35},
        metrics={"seg": [dice_metric], "order": ["accuracy"]},
    )
    if args.show_summary:
        model.summary()
    else:
        print(f"Model params: {model.count_params():,} (use --show-summary nếu muốn xem full summary)")

    ckpt = out_dir / "best_visible_order_teacher.keras"
    final_model_path = out_dir / "final_visible_order_teacher.keras"
    ckpt_dir = out_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    initial_epoch = 0
    if args.force_restart:
        print("[6v1] force restart: ignore old checkpoints")
    elif args.resume:
        last_epoch, last_ckpt = latest_weight_checkpoint(ckpt_dir)
        if last_ckpt is not None:
            model.load_weights(str(last_ckpt))
            initial_epoch = last_epoch
            print(f"[6v1] Resumed from {last_ckpt} -> initial_epoch={initial_epoch}")
        elif final_model_path.exists():
            loaded = keras.models.load_model(str(final_model_path), compile=False, safe_mode=False)
            model.set_weights(loaded.get_weights())
            print(f"[6v1] Loaded final model weights: {final_model_path}")
        elif ckpt.exists():
            loaded = keras.models.load_model(str(ckpt), compile=False, safe_mode=False)
            model.set_weights(loaded.get_weights())
            print(f"[6v1] Loaded best model weights: {ckpt}")
        else:
            print("[6v1] No checkpoint found, train from scratch")

    callbacks = [
        keras.callbacks.ModelCheckpoint(str(ckpt), monitor="val_seg_dice_metric", mode="max", save_best_only=True, verbose=0),
        keras.callbacks.ModelCheckpoint(str(ckpt_dir / "epoch_{epoch:03d}.weights.h5"), save_weights_only=True, save_freq="epoch", verbose=0),
        KeepLastCheckpoints(ckpt_dir, keep=args.max_keep_checkpoints),
        keras.callbacks.EarlyStopping(monitor="val_seg_dice_metric", mode="max", patience=args.patience, restore_best_weights=True, verbose=0),
        keras.callbacks.CSVLogger(str(out_dir / "train_visible_order_epoch_log.csv"), append=bool(args.resume and initial_epoch > 0)),
    ]
    fit_verbose = 1 if args.progress_mode == "keras" else 0
    if args.progress_mode != "keras":
        callbacks.insert(0, CleanProgressCallback(args.epochs, steps_per_epoch, args.progress_mode))

    if initial_epoch >= args.epochs:
        print(f"[6v1] Already reached epoch {initial_epoch}/{args.epochs}; skip training and evaluate best model.")
        hist = None
    else:
        hist = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=args.epochs,
            initial_epoch=initial_epoch,
            callbacks=callbacks,
            verbose=fit_verbose,
        )
        if hist is not None and hist.history:
            save_history(hist, out_dir)
    model.save(str(final_model_path))
    print(f"[6v1] Saved best model : {ckpt}")
    print(f"[6v1] Saved final model: {out_dir / 'final_visible_order_teacher.keras'}")
    print(f"[6v1] Saved history    : {out_dir / 'history_visible_order.csv'}")

    if not ckpt.exists():
        model.save(str(ckpt))
    best = keras.models.load_model(str(ckpt), compile=False, safe_mode=False)
    metrics = evaluate_real_test(best, dataset_dir, out_dir, batch_size=args.batch_size)
    print(json.dumps(metrics, indent=2))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile 9v2_train_missing_completion_teacher_student.py
"""
9v2_train_missing_completion_teacher_student.py
Model 2: missing-part/complement prediction.

Purpose:
- After 6v1 predicts visible A/B/C and top order, this stage learns the hidden part
  of the chromosome that is under the overlap.
- Teacher receives order maps as extra channels, so it is the stronger branch.
- Student receives no explicit order maps, so it must infer the missing branch from image + visible masks.
- Both predict 2 channels: gap_A and gap_B.
  gap_A = C only when A is under B, else empty.
  gap_B = C only when B is under A, else empty.

Dataset expected:
    dataset/train|val|test|real_test/images
    dataset/.../visible_A, visible_B, masks_C, gap_A, gap_B, order_labels.csv

Outputs:
    results/missing_completion/
        best_missing_teacher.keras
        best_missing_student.keras
        metrics_real_test_compare.json
        missing_compare_showcase_60.png
        history csv/json/curves
"""
from __future__ import annotations

import argparse, csv, json, os, random, math, time, re
from pathlib import Path
from typing import Dict

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
import tensorflow as tf
from tensorflow import keras
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

tf.get_logger().setLevel("ERROR")
try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
except Exception:
    pass
from tensorflow.keras import layers

IMG_SIZE = 256
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def read_labels(split_dir: Path) -> Dict[str, int]:
    csv_path = split_dir / "order_labels.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing {csv_path}")
    labels = {}
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f): labels[row["filename"]] = int(row["top_class"])
    return labels


def get_split_lists(dataset_dir: Path, split: str):
    split_dir = dataset_dir / split
    labels = read_labels(split_dir)
    rows = []
    for p in sorted((split_dir / "images").glob("*.png")):
        n = p.name
        needed = ["visible_A", "visible_B", "masks_C", "gap_A", "gap_B"]
        if n in labels and all((split_dir / sub / n).exists() for sub in needed):
            rows.append((str(p), str(split_dir/"visible_A"/n), str(split_dir/"visible_B"/n), str(split_dir/"masks_C"/n), str(split_dir/"gap_A"/n), str(split_dir/"gap_B"/n), labels[n], n))
    if not rows: raise FileNotFoundError(f"No valid samples in {split_dir}")
    return rows


def read_gray(path):
    img = tf.io.read_file(path); img = tf.image.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear")
    img.set_shape([IMG_SIZE, IMG_SIZE, 1]); return img


def read_mask(path):
    m = tf.io.read_file(path); m = tf.image.decode_png(m, channels=1)
    m = tf.image.resize(m, (IMG_SIZE, IMG_SIZE), method="nearest")
    m = tf.cast(m > 127, tf.float32); m.set_shape([IMG_SIZE, IMG_SIZE, 1]); return m


def load_teacher(img_p, va_p, vb_p, c_p, ga_p, gb_p, order, name):
    gray, va, vb, c = read_gray(img_p), read_mask(va_p), read_mask(vb_p), read_mask(c_p)
    ga, gb = read_mask(ga_p), read_mask(gb_p)
    order = tf.cast(order, tf.int32)
    top_a = tf.ones_like(gray) * tf.cast(tf.equal(order, 0), tf.float32)
    top_b = tf.ones_like(gray) * tf.cast(tf.equal(order, 1), tf.float32)
    x = tf.concat([gray, va, vb, c, top_a, top_b], axis=-1)
    y = tf.concat([ga, gb], axis=-1)
    x.set_shape([IMG_SIZE, IMG_SIZE, 6]); y.set_shape([IMG_SIZE, IMG_SIZE, 2])
    return x, y


def load_student(img_p, va_p, vb_p, c_p, ga_p, gb_p, order, name):
    gray, va, vb, c = read_gray(img_p), read_mask(va_p), read_mask(vb_p), read_mask(c_p)
    ga, gb = read_mask(ga_p), read_mask(gb_p)
    x = tf.concat([gray, va, vb, c], axis=-1)
    y = tf.concat([ga, gb], axis=-1)
    x.set_shape([IMG_SIZE, IMG_SIZE, 4]); y.set_shape([IMG_SIZE, IMG_SIZE, 2])
    return x, y


def augment(x, y):
    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_left_right(x); y = tf.image.flip_left_right(y)
    if tf.random.uniform(()) > 0.5:
        x = tf.image.flip_up_down(x); y = tf.image.flip_up_down(y)
    return x, y


def make_ds(dataset_dir: Path, split: str, role: str, batch: int, shuffle=False, do_aug=False):
    rows = get_split_lists(dataset_dir, split)
    cols = list(zip(*rows))
    ds = tf.data.Dataset.from_tensor_slices(cols)
    if shuffle:
        ds = ds.shuffle(min(len(rows), 4096), seed=SEED, reshuffle_each_iteration=True)
    loader = load_teacher if role == "teacher" else load_student
    ds = ds.map(loader, num_parallel_calls=tf.data.AUTOTUNE)
    if do_aug: ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch).prefetch(1), len(rows)


def conv(x, f, drop=0):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(f, 3, padding="same", use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    if drop: x = layers.SpatialDropout2D(drop)(x)
    return x


def build_unet(input_channels: int, base=32, name="missing_completion_unet"):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, input_channels))
    s1 = conv(inp, base); p1 = layers.MaxPooling2D()(s1)
    s2 = conv(p1, base*2); p2 = layers.MaxPooling2D()(s2)
    s3 = conv(p2, base*4, .05); p3 = layers.MaxPooling2D()(s3)
    s4 = conv(p3, base*8, .10); p4 = layers.MaxPooling2D()(s4)
    b = conv(p4, base*16, .15)
    x = layers.Conv2DTranspose(base*8, 2, 2, padding="same")(b); x = layers.Concatenate()([x,s4]); x = conv(x, base*8, .10)
    x = layers.Conv2DTranspose(base*4, 2, 2, padding="same")(x); x = layers.Concatenate()([x,s3]); x = conv(x, base*4, .05)
    x = layers.Conv2DTranspose(base*2, 2, 2, padding="same")(x); x = layers.Concatenate()([x,s2]); x = conv(x, base*2)
    x = layers.Conv2DTranspose(base, 2, 2, padding="same")(x); x = layers.Concatenate()([x,s1]); x = conv(x, base)
    out = layers.Conv2D(2, 1, activation="sigmoid", dtype="float32", name="gap_A_gap_B")(x)
    return keras.Model(inp, out, name=name)


def dice_loss(y_true, y_pred):
    inter = tf.reduce_sum(y_true * y_pred, axis=[1,2])
    den = tf.reduce_sum(y_true + y_pred, axis=[1,2])
    dice = (2*inter + 1e-6) / (den + 1e-6)
    return 1.0 - tf.reduce_mean(dice)


def gap_loss(y_true, y_pred):
    bce = keras.backend.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)


def gap_dice(y_true, y_pred):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * y_pred, axis=[1,2])
    den = tf.reduce_sum(y_true + y_pred, axis=[1,2])
    return tf.reduce_mean((2*inter + 1e-6)/(den + 1e-6))


def save_history(hist, out_dir: Path, name: str):
    data = {k:[float(x) for x in v] for k,v in hist.history.items()}
    with open(out_dir / f"{name}_history.json", "w", encoding="utf-8") as f: json.dump(data, f, indent=2)
    keys = list(data.keys())
    with open(out_dir / f"{name}_history.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["epoch"] + keys)
        for i in range(len(next(iter(data.values())))): w.writerow([i+1] + [data[k][i] for k in keys])
    for key, title in [("loss", "Loss"), ("gap_dice", "Gap Dice")]:
        plt.figure(figsize=(7,4))
        if key in data: plt.plot(data[key], label="train")
        if "val_"+key in data: plt.plot(data["val_"+key], label="val")
        plt.title(f"{name} {title}"); plt.xlabel("Epoch"); plt.ylabel(title); plt.legend(); plt.tight_layout()
        plt.savefig(out_dir / f"{name}_{key}_curve.png", dpi=160); plt.close()


def np_gray(path):
    return (np.array(Image.open(path).convert("L").resize((IMG_SIZE,IMG_SIZE))).astype(np.float32)/255.0)[...,None]

def np_mask(path):
    return (np.array(Image.open(path).convert("L").resize((IMG_SIZE,IMG_SIZE))).astype(np.float32) > 127).astype(np.float32)


def make_np_inputs(dataset_dir: Path, split="real_test"):
    rows = get_split_lists(dataset_dir, split)
    X_teacher, X_student, Y, names = [], [], [], []
    for img_p, va_p, vb_p, c_p, ga_p, gb_p, order, name in rows:
        gray, va, vb, c = np_gray(img_p), np_mask(va_p)[...,None], np_mask(vb_p)[...,None], np_mask(c_p)[...,None]
        top_a = np.ones_like(gray) * (1.0 if int(order) == 0 else 0.0)
        top_b = np.ones_like(gray) * (1.0 if int(order) == 1 else 0.0)
        X_teacher.append(np.concatenate([gray, va, vb, c, top_a, top_b], axis=-1))
        X_student.append(np.concatenate([gray, va, vb, c], axis=-1))
        Y.append(np.stack([np_mask(ga_p), np_mask(gb_p)], axis=-1))
        names.append(name)
    return np.stack(X_teacher).astype(np.float32), np.stack(X_student).astype(np.float32), np.stack(Y).astype(np.float32), names


def eval_compare(teacher, student, dataset_dir: Path, out_dir: Path, batch_size=16, max_show=60):
    Xt, Xs, Y, names = make_np_inputs(dataset_dir, "real_test")
    print(f"[9v2][eval] Predicting real_test: {len(names)} images ...")
    Pt = (teacher.predict(Xt, batch_size=batch_size, verbose=0) >= 0.5).astype(np.float32)
    Ps = (student.predict(Xs, batch_size=batch_size, verbose=0) >= 0.5).astype(np.float32)

    def dice_np(y, p):
        inter = np.sum(y*p, axis=(1,2))
        den = np.sum(y+p, axis=(1,2))
        d = (2*inter + 1e-7) / (den + 1e-7)
        return d
    dt = dice_np(Y, Pt); ds = dice_np(Y, Ps)
    mean_t_ch = dt.mean(axis=0); mean_s_ch = ds.mean(axis=0)
    mean_t, mean_s = float(dt.mean()), float(ds.mean())
    best = "teacher" if mean_t >= mean_s else "student"
    metrics = {
        "real_test_count": len(names),
        "teacher_mean_gap_dice_percent": mean_t*100,
        "student_mean_gap_dice_percent": mean_s*100,
        "teacher_gap_A_dice_percent": float(mean_t_ch[0]*100),
        "teacher_gap_B_dice_percent": float(mean_t_ch[1]*100),
        "student_gap_A_dice_percent": float(mean_s_ch[0]*100),
        "student_gap_B_dice_percent": float(mean_s_ch[1]*100),
        "best_model_on_real_test": best,
    }
    with open(out_dir / "metrics_real_test_compare.json", "w", encoding="utf-8") as f: json.dump(metrics, f, indent=2)

    # bar chart
    plt.figure(figsize=(6,4))
    x = np.arange(2); width = 0.35
    plt.bar(x-width/2, mean_t_ch*100, width, label="Teacher")
    plt.bar(x+width/2, mean_s_ch*100, width, label="Student")
    plt.xticks(x, ["gap_A", "gap_B"]); plt.ylabel("Dice %"); plt.title("Real test missing-gap Dice")
    plt.legend(); plt.tight_layout(); plt.savefig(out_dir / "teacher_student_gap_dice_compare.png", dpi=180); plt.close()

    # showcase 60: overlap | GT gapA/gapB | teacher gap | student gap
    show = min(max_show, len(names)); cols=5
    fig, axes = plt.subplots(show, cols, figsize=(cols*2.2, max(9, show*1.25)))
    if show == 1: axes = np.expand_dims(axes,0)
    for i in range(show):
        gray = Xs[i,...,0]
        gt_union = np.maximum(Y[i,...,0], Y[i,...,1])
        t_union = np.maximum(Pt[i,...,0], Pt[i,...,1])
        s_union = np.maximum(Ps[i,...,0], Ps[i,...,1])
        err = np.abs(gt_union - (t_union if best == "teacher" else s_union))
        imgs = [gray, gt_union, t_union, s_union, err]
        titles = ["Overlap", "GT missing", "Teacher", "Student", "Best err"]
        for j in range(cols):
            axes[i,j].imshow(imgs[j], cmap="gray"); axes[i,j].axis("off")
            if i == 0: axes[i,j].set_title(titles[j], fontsize=8)
    plt.tight_layout(); plt.savefig(out_dir / "missing_compare_showcase_60.png", dpi=180); plt.close()
    return metrics


def checkpoint_epoch(path: Path) -> int:
    m = re.search(r"epoch_(\d+)\.weights\.h5$", path.name)
    return int(m.group(1)) if m else -1


def latest_weight_checkpoint(ckpt_dir: Path):
    files = sorted(ckpt_dir.glob("epoch_*.weights.h5"), key=checkpoint_epoch)
    files = [p for p in files if checkpoint_epoch(p) > 0]
    if not files:
        return 0, None
    p = files[-1]
    return checkpoint_epoch(p), p


class KeepLastCheckpoints(keras.callbacks.Callback):
    def __init__(self, ckpt_dir: Path, keep: int = 3):
        super().__init__()
        self.ckpt_dir = Path(ckpt_dir)
        self.keep = int(max(1, keep))

    def on_epoch_end(self, epoch, logs=None):
        files = sorted(self.ckpt_dir.glob("epoch_*.weights.h5"), key=checkpoint_epoch)
        for p in files[:-self.keep]:
            try:
                p.unlink()
            except Exception:
                pass


class CleanProgressCallback(keras.callbacks.Callback):
    def __init__(self, role: str, total_epochs: int, steps_per_epoch: int, mode: str = "line"):
        super().__init__()
        self.role = role
        self.total_epochs = int(total_epochs)
        self.steps_per_epoch = int(max(1, steps_per_epoch))
        self.mode = mode
        self.pbar = None
        self.t0 = None

    @staticmethod
    def _get(logs, key, default=None):
        if not logs:
            return default
        try:
            return float(logs.get(key, default))
        except Exception:
            return default

    @staticmethod
    def _fmt(v, percent=False):
        if v is None:
            return "-"
        return f"{v*100:.2f}%" if percent else f"{v:.4f}"

    def on_epoch_begin(self, epoch, logs=None):
        self.t0 = time.time()
        desc = f"[9v2][{self.role}] Epoch {epoch+1:03d}/{self.total_epochs:03d}"
        if self.mode == "tqdm" and tqdm is not None:
            self.pbar = tqdm(total=self.steps_per_epoch, desc=desc, unit="batch", leave=False, dynamic_ncols=True, mininterval=0.5)
        else:
            print(desc)

    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}
        if self.pbar is not None:
            self.pbar.update(1)
            self.pbar.set_postfix({"loss": self._fmt(self._get(logs, "loss")), "dice": self._fmt(self._get(logs, "gap_dice"), percent=True)})
        elif self.mode == "line":
            step = batch + 1
            if step == 1 or step == self.steps_per_epoch or step % max(1, self.steps_per_epoch // 5) == 0:
                pct = step / self.steps_per_epoch * 100
                print(f"  step {step:03d}/{self.steps_per_epoch:03d} ({pct:5.1f}%) | loss={self._fmt(self._get(logs,'loss'))} | dice={self._fmt(self._get(logs,'gap_dice'), percent=True)}")

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if self.pbar is not None:
            self.pbar.close(); self.pbar = None
        sec = time.time() - self.t0 if self.t0 else 0.0
        print(
            f"[9v2][{self.role}][{epoch+1:03d}/{self.total_epochs:03d}] {sec:6.1f}s | "
            f"loss={self._fmt(self._get(logs,'loss'))} | val_loss={self._fmt(self._get(logs,'val_loss'))} | "
            f"dice={self._fmt(self._get(logs,'gap_dice'), percent=True)} | val_dice={self._fmt(self._get(logs,'val_gap_dice'), percent=True)}",
            flush=True,
        )


def train_one(role, dataset_dir, out_dir, epochs, batch, base, lr, patience, resume=False, force_restart=False, progress_mode="line", max_keep_checkpoints=3):
    channels = 6 if role == "teacher" else 4
    model = build_unet(channels, base=base, name=f"missing_{role}")
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=gap_loss, metrics=[gap_dice])
    train_ds, nt = make_ds(dataset_dir, "train", role, batch, True, True)
    val_ds, nv = make_ds(dataset_dir, "val", role, batch)
    steps_per_epoch = math.ceil(nt / batch)
    val_steps = math.ceil(nv / batch)
    print("="*80)
    print(f"9v2 MISSING COMPLETION {role.upper()} TRAINING")
    print("="*80)
    print(f"Dataset: train={nt}, val={nv}")
    print(f"Batch={batch} | steps/epoch={steps_per_epoch} | val_steps={val_steps}")
    print(f"Epochs={epochs} | patience={patience} | base_filters={base} | lr={lr}")

    ckpt = out_dir / f"best_missing_{role}.keras"
    final_path = out_dir / f"final_missing_{role}.keras"
    ckpt_dir = out_dir / f"checkpoints_{role}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    initial_epoch = 0
    if force_restart:
        print(f"[9v2][{role}] force restart: ignore old checkpoints")
    elif resume:
        last_epoch, last_ckpt = latest_weight_checkpoint(ckpt_dir)
        if last_ckpt is not None:
            model.load_weights(str(last_ckpt))
            initial_epoch = last_epoch
            print(f"[9v2][{role}] Resumed from {last_ckpt} -> initial_epoch={initial_epoch}")
        elif final_path.exists():
            loaded = keras.models.load_model(str(final_path), compile=False, safe_mode=False)
            model.set_weights(loaded.get_weights())
            print(f"[9v2][{role}] Loaded final model weights: {final_path}")
        elif ckpt.exists():
            loaded = keras.models.load_model(str(ckpt), compile=False, safe_mode=False)
            model.set_weights(loaded.get_weights())
            print(f"[9v2][{role}] Loaded best model weights: {ckpt}")
        else:
            print(f"[9v2][{role}] No checkpoint found, train from scratch")

    callbacks = [
        keras.callbacks.ModelCheckpoint(str(ckpt), monitor="val_gap_dice", mode="max", save_best_only=True, verbose=0),
        keras.callbacks.ModelCheckpoint(str(ckpt_dir / "epoch_{epoch:03d}.weights.h5"), save_weights_only=True, save_freq="epoch", verbose=0),
        KeepLastCheckpoints(ckpt_dir, keep=max_keep_checkpoints),
        keras.callbacks.EarlyStopping(monitor="val_gap_dice", mode="max", patience=patience, restore_best_weights=True, verbose=0),
        keras.callbacks.CSVLogger(str(out_dir / f"{role}_epoch_log.csv"), append=bool(resume and initial_epoch > 0)),
    ]
    fit_verbose = 1 if progress_mode == "keras" else 0
    if progress_mode != "keras":
        callbacks.insert(0, CleanProgressCallback(role, epochs, steps_per_epoch, progress_mode))

    if initial_epoch >= epochs:
        print(f"[9v2][{role}] Already reached epoch {initial_epoch}/{epochs}; skip training.")
        hist = None
    else:
        hist = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            initial_epoch=initial_epoch,
            callbacks=callbacks,
            verbose=fit_verbose,
        )
        if hist is not None and hist.history:
            save_history(hist, out_dir, role)
    model.save(str(final_path))
    if not ckpt.exists():
        model.save(str(ckpt))
    print(f"[9v2][{role}] Saved best : {ckpt}")
    print(f"[9v2][{role}] Saved final: {final_path}")
    return keras.models.load_model(str(ckpt), compile=False, safe_mode=False)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset-dir", default="dataset")
    ap.add_argument("--results-dir", default="results")
    ap.add_argument("--epochs", type=int, default=80)
    ap.add_argument("--batch-size", type=int, default=24)
    ap.add_argument("--base-filters", type=int, default=32)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--patience", type=int, default=10)
    ap.add_argument("--skip-teacher", action="store_true")
    ap.add_argument("--skip-student", action="store_true")
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--force-restart", action="store_true")
    ap.add_argument("--progress-mode", choices=["tqdm", "line", "keras"], default="line")
    ap.add_argument("--max-keep-checkpoints", type=int, default=3)
    args = ap.parse_args()

    dataset_dir = Path(args.dataset_dir)
    out_dir = Path(args.results_dir) / "missing_completion"
    out_dir.mkdir(parents=True, exist_ok=True)

    if not args.skip_teacher:
        teacher = train_one("teacher", dataset_dir, out_dir, args.epochs, args.batch_size, args.base_filters, args.lr, args.patience, args.resume, args.force_restart, args.progress_mode, args.max_keep_checkpoints)
    else:
        teacher = keras.models.load_model(str(out_dir / "best_missing_teacher.keras"), compile=False, safe_mode=False)

    if not args.skip_student:
        student = train_one("student", dataset_dir, out_dir, args.epochs, args.batch_size, max(16, args.base_filters//2), args.lr, args.patience, args.resume, args.force_restart, args.progress_mode, args.max_keep_checkpoints)
    else:
        student = keras.models.load_model(str(out_dir / "best_missing_student.keras"), compile=False, safe_mode=False)

    metrics = eval_compare(teacher, student, dataset_dir, out_dir, batch_size=args.batch_size)
    print(json.dumps(metrics, indent=2))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile 10v2_evaluate_full_missing_pipeline.py
"""
10v2_evaluate_full_missing_pipeline.py
Run the complete missing-part prediction pipeline on the 1,000-image real_test split.

Pipeline:
1. Load 6v1 visible/order model.
2. Predict visible_A, visible_B, C, and top order for real_test images.
3. Feed those 6v1 predictions into the missing-completion Teacher and Student.
4. Compare Teacher vs Student on gap_A/gap_B and reconstructed full A/B.
5. Save statistics, heatmaps, predicted masks, and a 60-case showcase.

Expected trained files:
    results/visible_order/best_visible_order_teacher.keras
    results/missing_completion/best_missing_teacher.keras
    results/missing_completion/best_missing_student.keras
"""
from __future__ import annotations

import argparse, csv, json, os
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
import tensorflow as tf
from tensorflow import keras
tf.get_logger().setLevel("ERROR")
try:
    import absl.logging
    absl.logging.set_verbosity(absl.logging.ERROR)
except Exception:
    pass

IMG_SIZE = 256
EPS = 1e-7


def read_labels(split_dir: Path):
    label_csv = split_dir / "order_labels.csv"
    if not label_csv.exists():
        raise FileNotFoundError(f"Missing {label_csv}")
    labels = {}
    with open(label_csv, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            labels[row["filename"]] = int(row["top_class"])
    return labels


def np_gray(path: Path):
    arr = np.array(Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
    return arr[..., None]


def np_mask(path: Path):
    arr = np.array(Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE))).astype(np.float32)
    return (arr > 127).astype(np.float32)


def load_real_test(dataset_dir: Path):
    split_dir = dataset_dir / "real_test"
    labels = read_labels(split_dir)
    image_paths = sorted((split_dir / "images").glob("*.png"))
    names = [p.name for p in image_paths if p.name in labels]
    if not names:
        raise FileNotFoundError(f"No real_test samples in {split_dir}")

    X = []
    y_visible = []
    y_full = []
    y_gap = []
    y_order = []
    for n in names:
        X.append(np_gray(split_dir / "images" / n))
        va = np_mask(split_dir / "visible_A" / n)
        vb = np_mask(split_dir / "visible_B" / n)
        c = np_mask(split_dir / "masks_C" / n)
        ma = np_mask(split_dir / "masks_A" / n)
        mb = np_mask(split_dir / "masks_B" / n)
        ga = np_mask(split_dir / "gap_A" / n)
        gb = np_mask(split_dir / "gap_B" / n)
        y_visible.append(np.stack([va, vb, c], axis=-1))
        y_full.append(np.stack([ma, mb], axis=-1))
        y_gap.append(np.stack([ga, gb], axis=-1))
        y_order.append(labels[n])

    return (
        np.stack(X).astype(np.float32),
        np.stack(y_visible).astype(np.float32),
        np.stack(y_full).astype(np.float32),
        np.stack(y_gap).astype(np.float32),
        np.array(y_order, dtype=np.int32),
        names,
    )



def unpack_visible_order_prediction(pred):
    """Return (seg_prob, order_prob) for Keras dict/list outputs."""
    if isinstance(pred, dict):
        return pred["seg"], pred["order"]
    if isinstance(pred, (list, tuple)):
        seg = None; order = None
        for item in pred:
            arr = np.asarray(item)
            if arr.ndim == 4 and arr.shape[-1] == 3:
                seg = item
            elif arr.ndim == 2 and arr.shape[-1] == 2:
                order = item
        if seg is not None and order is not None:
            return seg, order
    raise ValueError(f"Cannot unpack visible/order prediction outputs: {type(pred)}")


def make_c_rule_candidates(pred_visible, pred_gap):
    """
    Build two explicit A/B reconstruction hypotheses from predicted C.
    candidate_A_ON_TOP: A owns C, B receives missing C/gap.
    candidate_B_ON_TOP: B owns C, A receives missing C/gap.
    """
    va = pred_visible[..., 0]
    vb = pred_visible[..., 1]
    c = pred_visible[..., 2]
    ga = pred_gap[..., 0]
    gb = pred_gap[..., 1]

    a_top_A = np.maximum(va, c)
    a_top_B = np.maximum(vb, gb)

    b_top_A = np.maximum(va, ga)
    b_top_B = np.maximum(vb, c)

    cand_a_top = np.stack([a_top_A, a_top_B], axis=-1).astype(np.float32)
    cand_b_top = np.stack([b_top_A, b_top_B], axis=-1).astype(np.float32)
    return cand_a_top, cand_b_top


def dice_per_channel(y_true, y_pred):
    inter = np.sum(y_true * y_pred, axis=(0, 1, 2))
    den = np.sum(y_true + y_pred, axis=(0, 1, 2))
    return (2 * inter + EPS) / (den + EPS)


def dice_per_image_channel(y_true, y_pred):
    inter = np.sum(y_true * y_pred, axis=(1, 2))
    den = np.sum(y_true + y_pred, axis=(1, 2))
    return (2 * inter + EPS) / (den + EPS)


def pixel_acc(y_true, y_pred):
    return float(np.mean(y_true == y_pred))


def build_missing_inputs(X_gray, pred_visible, pred_order):
    top_a = (pred_order == 0).astype(np.float32)[:, None, None, None]
    top_b = (pred_order == 1).astype(np.float32)[:, None, None, None]
    top_a_map = np.ones_like(X_gray) * top_a
    top_b_map = np.ones_like(X_gray) * top_b
    X_teacher = np.concatenate([X_gray, pred_visible, top_a_map, top_b_map], axis=-1).astype(np.float32)
    X_student = np.concatenate([X_gray, pred_visible], axis=-1).astype(np.float32)
    return X_teacher, X_student


def reconstruct_full(pred_visible, pred_gap, pred_order):
    """
    pred_visible: [N,H,W,3] = visible_A, visible_B, C
    pred_gap    : [N,H,W,2] = gap_A, gap_B
    pred_order  : 0 A_ON_TOP, 1 B_ON_TOP
    Returns predicted full A/B masks using both C decision and gap completion.
    """
    va = pred_visible[..., 0]
    vb = pred_visible[..., 1]
    c = pred_visible[..., 2]
    ga = pred_gap[..., 0]
    gb = pred_gap[..., 1]

    rec_a = np.maximum(va, ga)
    rec_b = np.maximum(vb, gb)

    # Enforce the C-to-top rule from the order classifier.
    for i, order in enumerate(pred_order):
        if int(order) == 0:  # A_ON_TOP => A owns C visually, B has missing C
            rec_a[i] = np.maximum(rec_a[i], c[i])
            rec_b[i] = np.maximum(rec_b[i], gb[i])
        else:                # B_ON_TOP => B owns C visually, A has missing C
            rec_b[i] = np.maximum(rec_b[i], c[i])
            rec_a[i] = np.maximum(rec_a[i], ga[i])

    return np.stack([rec_a, rec_b], axis=-1).astype(np.float32)


def save_mask(arr, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((arr > 0.5).astype(np.uint8) * 255).save(path)


def plot_confusion(y_true, y_pred, out_path: Path):
    conf = np.zeros((2, 2), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        conf[int(t), int(p)] += 1
    conf_norm = conf.astype(np.float64) / np.maximum(conf.sum(axis=1, keepdims=True), 1)
    plt.figure(figsize=(5, 4))
    plt.imshow(conf_norm * 100)
    plt.xticks([0, 1], ["A_TOP", "B_TOP"])
    plt.yticks([0, 1], ["A_TOP", "B_TOP"])
    plt.xlabel("Predicted")
    plt.ylabel("Ground truth")
    plt.title("6v1 Order prediction confusion (%)")
    for i in range(2):
        for j in range(2):
            plt.text(j, i, f"{conf_norm[i, j] * 100:.1f}%", ha="center", va="center")
    plt.colorbar(label="Percent")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()
    return conf


def make_metric_bar(metrics, out_path: Path):
    labels = ["Gap mean", "Full A/B mean"]
    teacher = [metrics["teacher_mean_gap_dice_percent"], metrics["teacher_mean_full_AB_dice_percent"]]
    student = [metrics["student_mean_gap_dice_percent"], metrics["student_mean_full_AB_dice_percent"]]
    x = np.arange(len(labels))
    width = 0.35
    plt.figure(figsize=(7, 4))
    plt.bar(x - width/2, teacher, width, label="Teacher")
    plt.bar(x + width/2, student, width, label="Student")
    plt.xticks(x, labels)
    plt.ylabel("Dice (%)")
    plt.title("Full pipeline: Teacher vs Student on real_test")
    plt.legend()
    for i, v in enumerate(teacher):
        plt.text(i - width/2, v + 0.5, f"{v:.1f}", ha="center", fontsize=8)
    for i, v in enumerate(student):
        plt.text(i + width/2, v + 0.5, f"{v:.1f}", ha="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def make_showcase(X, y_full, y_gap, pred_visible, pred_order, p_gap_t, p_gap_s, rec_t, rec_s, best_name, names, out_path: Path, max_show=60):
    n = min(max_show, len(names))
    cols = 8
    fig, axes = plt.subplots(n, cols, figsize=(cols * 1.75, max(10, n * 1.10)))
    if n == 1:
        axes = np.expand_dims(axes, 0)

    best_rec = rec_t if best_name == "teacher" else rec_s
    best_gap = p_gap_t if best_name == "teacher" else p_gap_s

    titles = ["Overlap", "Pred A vis", "Pred B vis", "Pred C/order", "GT missing", "Teacher missing", "Student missing", "Best full A|B"]
    for i in range(n):
        gray = X[i, ..., 0]
        pred_c = pred_visible[i, ..., 2]
        gt_gap_union = np.maximum(y_gap[i, ..., 0], y_gap[i, ..., 1])
        t_gap_union = np.maximum(p_gap_t[i, ..., 0], p_gap_t[i, ..., 1])
        s_gap_union = np.maximum(p_gap_s[i, ..., 0], p_gap_s[i, ..., 1])
        full_ab = np.concatenate([best_rec[i, ..., 0], best_rec[i, ..., 1]], axis=1)
        order_txt = "A_TOP" if pred_order[i] == 0 else "B_TOP"

        imgs = [gray, pred_visible[i, ..., 0], pred_visible[i, ..., 1], pred_c, gt_gap_union, t_gap_union, s_gap_union, full_ab]
        for j, img in enumerate(imgs):
            axes[i, j].imshow(img, cmap="gray")
            axes[i, j].axis("off")
            if i == 0:
                axes[i, j].set_title(titles[j], fontsize=7)
        axes[i, 3].text(2, 12, order_txt, color="yellow", fontsize=6, bbox=dict(facecolor="black", alpha=0.4, pad=1))
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset-dir", default="dataset")
    ap.add_argument("--results-dir", default="results")
    ap.add_argument("--batch-size", type=int, default=40)
    ap.add_argument("--max-showcase", type=int, default=60)
    args = ap.parse_args()

    dataset_dir = Path(args.dataset_dir)
    results_dir = Path(args.results_dir)
    visible_model_path = results_dir / "visible_order" / "best_visible_order_teacher.keras"
    teacher_path = results_dir / "missing_completion" / "best_missing_teacher.keras"
    student_path = results_dir / "missing_completion" / "best_missing_student.keras"

    for p in [visible_model_path, teacher_path, student_path]:
        if not p.exists():
            raise FileNotFoundError(f"Missing trained model: {p}")

    out_dir = results_dir / "full_pipeline_real_test"
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Loading data...")
    X, y_visible, y_full, y_gap, y_order, names = load_real_test(dataset_dir)
    print(f"real_test samples: {len(names)} | Batch={args.batch_size}")

    print("Loading models from exact paths only, no recursive model search...")
    visible_model = keras.models.load_model(str(visible_model_path), compile=False, safe_mode=False)
    teacher = keras.models.load_model(str(teacher_path), compile=False, safe_mode=False)
    student = keras.models.load_model(str(student_path), compile=False, safe_mode=False)

    print("Step 1/3: Predict visible A/B/C and order with 6v1...")
    vo_pred = visible_model.predict(X, batch_size=args.batch_size, verbose=0)
    pred_visible_prob, pred_order_prob = unpack_visible_order_prediction(vo_pred)
    pred_visible = (pred_visible_prob >= 0.5).astype(np.float32)
    pred_order = np.argmax(pred_order_prob, axis=1).astype(np.int32)

    X_teacher, X_student = build_missing_inputs(X, pred_visible, pred_order)

    print("Step 2/3: Predict missing gaps with Teacher and Student from 6v1 outputs...")
    pred_gap_teacher = (teacher.predict(X_teacher, batch_size=args.batch_size, verbose=0) >= 0.5).astype(np.float32)
    pred_gap_student = (student.predict(X_student, batch_size=args.batch_size, verbose=0) >= 0.5).astype(np.float32)

    # Explicit C-rule hypotheses for report/debugging.
    # These represent the two possible interpretations of C before the order classifier chooses one.
    cand_teacher_A_top, cand_teacher_B_top = make_c_rule_candidates(pred_visible, pred_gap_teacher)
    cand_student_A_top, cand_student_B_top = make_c_rule_candidates(pred_visible, pred_gap_student)

    rec_teacher = reconstruct_full(pred_visible, pred_gap_teacher, pred_order)
    rec_student = reconstruct_full(pred_visible, pred_gap_student, pred_order)

    print("Step 3/3: Metrics and visualization...")
    d_vis = dice_per_channel(y_visible, pred_visible)
    d_gap_t = dice_per_channel(y_gap, pred_gap_teacher)
    d_gap_s = dice_per_channel(y_gap, pred_gap_student)
    d_full_t = dice_per_channel(y_full, rec_teacher)
    d_full_s = dice_per_channel(y_full, rec_student)
    order_acc = float(np.mean(pred_order == y_order))

    mean_gap_t = float(np.mean(d_gap_t))
    mean_gap_s = float(np.mean(d_gap_s))
    mean_full_t = float(np.mean(d_full_t))
    mean_full_s = float(np.mean(d_full_s))
    best = "teacher" if mean_full_t >= mean_full_s else "student"

    metrics = {
        "real_test_count": int(len(names)),
        "visible_order_model": {
            "visible_A_dice_percent": float(d_vis[0] * 100),
            "visible_B_dice_percent": float(d_vis[1] * 100),
            "C_overlap_dice_percent": float(d_vis[2] * 100),
            "mean_visible_ABC_dice_percent": float(np.mean(d_vis) * 100),
            "order_accuracy_percent": float(order_acc * 100),
            "visible_pixel_accuracy_percent": pixel_acc(y_visible, pred_visible) * 100,
        },
        "teacher_from_6v1_outputs": {
            "gap_A_dice_percent": float(d_gap_t[0] * 100),
            "gap_B_dice_percent": float(d_gap_t[1] * 100),
            "mean_gap_dice_percent": float(mean_gap_t * 100),
            "full_A_dice_percent": float(d_full_t[0] * 100),
            "full_B_dice_percent": float(d_full_t[1] * 100),
            "mean_full_AB_dice_percent": float(mean_full_t * 100),
        },
        "student_from_6v1_outputs": {
            "gap_A_dice_percent": float(d_gap_s[0] * 100),
            "gap_B_dice_percent": float(d_gap_s[1] * 100),
            "mean_gap_dice_percent": float(mean_gap_s * 100),
            "full_A_dice_percent": float(d_full_s[0] * 100),
            "full_B_dice_percent": float(d_full_s[1] * 100),
            "mean_full_AB_dice_percent": float(mean_full_s * 100),
        },
        "teacher_mean_gap_dice_percent": float(mean_gap_t * 100),
        "student_mean_gap_dice_percent": float(mean_gap_s * 100),
        "teacher_mean_full_AB_dice_percent": float(mean_full_t * 100),
        "student_mean_full_AB_dice_percent": float(mean_full_s * 100),
        "best_model_by_full_AB_dice": best,
    }

    with open(out_dir / "metrics_full_pipeline_real_test.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    with open(out_dir / "metrics_full_pipeline_real_test.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["metric", "value"])
        def flatten(prefix, obj):
            for k, v in obj.items():
                if isinstance(v, dict):
                    yield from flatten(prefix + k + ".", v)
                else:
                    yield prefix + k, v
        for k, v in flatten("", metrics):
            w.writerow([k, v])

    conf = plot_confusion(y_order, pred_order, out_dir / "order_confusion_from_6v1_heatmap.png")
    metrics["visible_order_model"]["order_confusion_matrix"] = conf.tolist()
    make_metric_bar(metrics, out_dir / "teacher_student_full_pipeline_compare.png")
    make_showcase(X, y_full, y_gap, pred_visible, pred_order, pred_gap_teacher, pred_gap_student, rec_teacher, rec_student, best, names, out_dir / "full_pipeline_showcase_60.png", args.max_showcase)

    # Save predicted masks for all real_test samples.
    pred_root = out_dir / "predicted_masks"
    subdirs = [
        "visible_A", "visible_B", "C_overlap", "order_text",
        "teacher_gap_A", "teacher_gap_B", "teacher_full_A", "teacher_full_B",
        "student_gap_A", "student_gap_B", "student_full_A", "student_full_B",
        "candidate_teacher_A_ON_TOP_full_A", "candidate_teacher_A_ON_TOP_full_B",
        "candidate_teacher_B_ON_TOP_full_A", "candidate_teacher_B_ON_TOP_full_B",
        "candidate_student_A_ON_TOP_full_A", "candidate_student_A_ON_TOP_full_B",
        "candidate_student_B_ON_TOP_full_A", "candidate_student_B_ON_TOP_full_B",
    ]
    for s in subdirs:
        (pred_root / s).mkdir(parents=True, exist_ok=True)

    order_rows = []
    for i, n in enumerate(names):
        stem = Path(n).stem + ".png"
        save_mask(pred_visible[i, ..., 0], pred_root / "visible_A" / stem)
        save_mask(pred_visible[i, ..., 1], pred_root / "visible_B" / stem)
        save_mask(pred_visible[i, ..., 2], pred_root / "C_overlap" / stem)
        save_mask(pred_gap_teacher[i, ..., 0], pred_root / "teacher_gap_A" / stem)
        save_mask(pred_gap_teacher[i, ..., 1], pred_root / "teacher_gap_B" / stem)
        save_mask(rec_teacher[i, ..., 0], pred_root / "teacher_full_A" / stem)
        save_mask(rec_teacher[i, ..., 1], pred_root / "teacher_full_B" / stem)
        save_mask(pred_gap_student[i, ..., 0], pred_root / "student_gap_A" / stem)
        save_mask(pred_gap_student[i, ..., 1], pred_root / "student_gap_B" / stem)
        save_mask(rec_student[i, ..., 0], pred_root / "student_full_A" / stem)
        save_mask(rec_student[i, ..., 1], pred_root / "student_full_B" / stem)

        # Save explicit C-to-A/B hypotheses: A_ON_TOP and B_ON_TOP.
        save_mask(cand_teacher_A_top[i, ..., 0], pred_root / "candidate_teacher_A_ON_TOP_full_A" / stem)
        save_mask(cand_teacher_A_top[i, ..., 1], pred_root / "candidate_teacher_A_ON_TOP_full_B" / stem)
        save_mask(cand_teacher_B_top[i, ..., 0], pred_root / "candidate_teacher_B_ON_TOP_full_A" / stem)
        save_mask(cand_teacher_B_top[i, ..., 1], pred_root / "candidate_teacher_B_ON_TOP_full_B" / stem)
        save_mask(cand_student_A_top[i, ..., 0], pred_root / "candidate_student_A_ON_TOP_full_A" / stem)
        save_mask(cand_student_A_top[i, ..., 1], pred_root / "candidate_student_A_ON_TOP_full_B" / stem)
        save_mask(cand_student_B_top[i, ..., 0], pred_root / "candidate_student_B_ON_TOP_full_A" / stem)
        save_mask(cand_student_B_top[i, ..., 1], pred_root / "candidate_student_B_ON_TOP_full_B" / stem)

        order_rows.append({
            "filename": n,
            "gt_order": "A_ON_TOP" if int(y_order[i]) == 0 else "B_ON_TOP",
            "pred_order": "A_ON_TOP" if int(pred_order[i]) == 0 else "B_ON_TOP",
            "prob_A_ON_TOP": float(pred_order_prob[i, 0]),
            "prob_B_ON_TOP": float(pred_order_prob[i, 1]),
        })

    with open(pred_root / "predicted_order.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(order_rows[0].keys()))
        writer.writeheader(); writer.writerows(order_rows)

    # Update metrics JSON after adding confusion.
    with open(out_dir / "metrics_full_pipeline_real_test.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print(json.dumps(metrics, indent=2))
    print(f"Saved full pipeline results to: {out_dir}")


if __name__ == "__main__":
    main()


In [ ]:
# =========================
# CELL 5 — SYNTAX CHECK + BASIC DATA CHECK
# =========================
import os, py_compile, json
from pathlib import Path

os.chdir(PROJECT_DIR)
script_files = [
    "2v1_prepare_single_chromosomes.py",
    "3v1_generate_synthetic_masks.py",
    "4v1_preprocess_to_256.py",
    "5v1_split_data.py",
    "6v1_train_visible_order.py",
    "9v2_train_missing_completion_teacher_student.py",
    "10v2_evaluate_full_missing_pipeline.py",
]
for f in script_files:
    py_compile.compile(f, doraise=True)
    print("compile OK:", f)

required_source = PROJECT_DIR / "source_data" / "single_chromosomes"
assert required_source.exists(), f"Missing raw source folder: {required_source}"
print("Raw single chromosome count:")
os.system("find source_data/single_chromosomes -type f | wc -l")
print("Logic OK: scripts compile; source_data/single_chromosomes exists.")


In [ ]:
# =========================
# CELL 6 — RUN HELPER WITH LIVE OUTPUT
# =========================
import subprocess, time, os
from pathlib import Path

os.chdir(PROJECT_DIR)

def run_cmd(cmd, title=None):
    if title:
        print("
" + "="*90)
        print(title)
        print("="*90)
    print("$", cmd)
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    ret = p.wait()
    print(f"
[exit={ret}] elapsed={time.time()-t0:.1f}s")
    if ret != 0:
        raise RuntimeError(f"Command failed: {cmd}")


In [ ]:
# =========================
# CELL 7 — PREPARE SINGLE CHROMOSOMES
# =========================
run_cmd("python -u 2v1_prepare_single_chromosomes.py", "2v1 prepare single chromosomes")


In [ ]:
# =========================
# CELL 8 — GENERATE 5000 SYNTHETIC SAMPLES
# =========================
run_cmd(
    f"python -u 3v1_generate_synthetic_masks.py --num-samples {NUM_SAMPLES} --seed 42 --progress-every 100",
    "3v1 generate 5000 samples: full A/B + visible A/B + C + gap labels"
)
print("Generated counts:")
os.system("find generated_data/images -type f | wc -l")
os.system("find generated_data/masks_C -type f | wc -l")


In [ ]:
# =========================
# CELL 9 — PREPROCESS 256 + SPLIT 4000/1000
# =========================
run_cmd("python -u 4v1_preprocess_to_256.py", "4v1 resize/pad to 256")
run_cmd("python -u 5v1_split_data.py", "5v1 split: 2800 train, 600 val, 600 test, 1000 real_test")
print("Dataset counts:")
for split in ["train", "val", "test", "real_test"]:
    print(split, end=": ")
    os.system(f"find dataset/{split}/images -type f | wc -l")


In [ ]:
# =========================
# CELL 10 — TRAIN 6v1 VISIBLE A/B/C + ORDER, WITH DRIVE RESUME
# =========================
cmd = (
    f"python -u 6v1_train_visible_order.py "
    f"--dataset-dir dataset --results-dir '{DRIVE_RESULTS}' "
    f"--epochs {VISIBLE_EPOCHS} --batch-size {BATCH_SIZE} "
    f"--base-filters {VISIBLE_BASE_FILTERS} --lr {LR} --patience {PATIENCE} "
    f"--progress-mode line --resume --max-keep-checkpoints {MAX_KEEP_CHECKPOINTS}"
)
run_cmd(cmd, "6v1 train visible A/B/C + A_ON_TOP/B_ON_TOP with resume")


In [ ]:
# =========================
# CELL 11 — TRAIN 9v2 MISSING COMPLETION TEACHER + STUDENT, WITH DRIVE RESUME
# =========================
cmd = (
    f"python -u 9v2_train_missing_completion_teacher_student.py "
    f"--dataset-dir dataset --results-dir '{DRIVE_RESULTS}' "
    f"--epochs {MISSING_EPOCHS} --batch-size {BATCH_SIZE} "
    f"--base-filters {MISSING_BASE_FILTERS} --lr {LR} --patience {PATIENCE} "
    f"--progress-mode line --resume --max-keep-checkpoints {MAX_KEEP_CHECKPOINTS}"
)
run_cmd(cmd, "9v2 train missing completion Teacher + Student with resume")


In [ ]:
# =========================
# CELL 12 — FULL PIPELINE REAL_TEST 1000 IMAGES
# Exact model paths inside DRIVE_RESULTS, no recursive model search.
# =========================
cmd = (
    f"python -u 10v2_evaluate_full_missing_pipeline.py "
    f"--dataset-dir dataset --results-dir '{DRIVE_RESULTS}' "
    f"--batch-size {BATCH_SIZE} --max-showcase 60"
)
run_cmd(cmd, "10v2 evaluate full pipeline on 1000 real_test")


In [ ]:
# =========================
# CELL 13 — SUMMARY FOR REPORT
# =========================
import json, os
from pathlib import Path

paths = [
    DRIVE_RESULTS / "visible_order" / "metrics_visible_order_real_test.json",
    DRIVE_RESULTS / "missing_completion" / "metrics_real_test_compare.json",
    DRIVE_RESULTS / "full_pipeline_real_test" / "metrics_full_pipeline_real_test.json",
]
for p in paths:
    print("
" + "="*90)
    print(p)
    print("="*90)
    if p.exists():
        print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False))
    else:
        print("Missing:", p)

print("
Drive model files:")
os.system(f"find '{DRIVE_RESULTS}' -maxdepth 4 -name '*.keras' -printf '%p %k KB\n' | sort")
print("
Drive checkpoint files kept:")
os.system(f"find '{DRIVE_RESULTS}' -maxdepth 5 -name 'epoch_*.weights.h5' -printf '%p %k KB\n' | sort")
print("
Drive outputs:")
os.system(f"find '{DRIVE_RESULTS}' -maxdepth 4 -type f | grep -E 'json|csv|png|zip' | head -120")


## Output đúng sau bản v4

Trong Drive `nst_tach_results/results_v4_complete` sẽ có:

- `visible_order/best_visible_order_teacher.keras`, `final_visible_order_teacher.keras`
- `missing_completion/best_missing_teacher.keras`, `best_missing_student.keras`, final teacher/student
- checkpoint resume: chỉ giữ 3 checkpoint gần nhất cho 6v1, teacher, student
- `full_pipeline_real_test/metrics_full_pipeline_real_test.json`
- `full_pipeline_real_test/predicted_masks/` gồm full A/B của teacher và student, cộng các candidate A_ON_TOP/B_ON_TOP để kiểm tra logic C.
